# Research Objective: Do NBA teams with higher three point attempts and makes win more games on average?
Relevance: In the past years there has been a revolution in the NBA of phasing out mid-range jump shots in turn for more lay-ups and three point jumpshots. In theory, this should increase offensive efficiency because three pointers are only marginally more difficult than mid-range shots that are a few feet in, but are valued 1 point more than the mid-range shot. So a team that takes more threes and low-difficulty lay-ups should score more efficiently. I will be evaluating historical, season-long team data to evaluate the trend of more three point shooting and it's correlation to winning. The goal is that this analysis will provide insight on the success of the strategy and the optimal way of implementing it. Of course, far more than successful three point attempts goes into winning basketball games, but analyses of this nature should aid us in discovering the puzzle pieces to basketball strategy optimization.

# Data Source: I will be obtaining the data for this project using TeamYearByYearStats from the nba_stats api (https://pypi.org/project/nba_api/#description and https://github.com/swar/nba_api/tree/master).
This api makes official stats from the endpoints at www.NBA.com easily accessible and provides documentation for each of them. The variables I will be utilizing are TEAM_CITY, TEAM_NAME, YEAR, WINS, LOSSES, WIN_PCT, FG3M, FG3A, FG3_PCT. Once the data was filtered down to only years after the three point line was introduced into the NBA (1979) there were 1216 rows of data and alltemas_year_over_year.info() showed us that each column had 1216 non-null values. This shows that this data set is good on completeness. The data quality is also very sound given this comes from the official NBA website.

# Intended Approach:
I will begin with some exploratory data analysis (EDA)techniques to get an idea of the relationships between the variables of interest. This will involve calcualating a correlation matrix of the variables along with visualizing the relationships between total three point attempts/makes and total wins.

After the EDA phase, I will fit two simple linear regressions of total season win percentage on three point field goal percentage. One based on the entire dataset from 1982-2023 and one based on the modern NBA only including seasons from 2010-2023. This will give me a basic sense of how three point shooting correlates with winning in the NBA both historically and in the modern era. We will be able to see shifts in the success of implementing a three point centric strategy and how this evolved into the most optimized strategy available.

## Import packages and define project paths

This cell imports the packages used throughout the notebook and sets up project-relative paths. Instead of changing the working directory to Downloads, the notebook looks for `Fulk_Proj_FinalReport.ipynb` and creates a `data/` folder in that same project folder. This keeps the notebook portable if the project folder is moved or shared.


In [ ]:

import numpy as np
import time
import random
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

from requests.exceptions import ReadTimeout, ConnectionError

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from nba_api.stats.static import teams
from nba_api.stats.endpoints import teamyearbyyearstats
from nba_api.stats.library.http import STATS_HEADERS


# Project-relative paths
NOTEBOOK_NAME = "The Three-Point Effect.ipynb"

def find_project_root(anchor_file=NOTEBOOK_NAME):
    """
    Find the project folder by looking for this notebook in the current
    working directory or one of its parent folders.
    """
    current_dir = Path.cwd().resolve()

    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / anchor_file).exists():
            return candidate

    # Fallback: use the current working directory if the notebook file
    # cannot be found from where Jupyter was launched.
    return current_dir


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
TEAM_CACHE_DIR = DATA_DIR / "nba_teamyearbyyear_cache"

HORNETS_CSV = DATA_DIR / "hornets_teamyearbyyear.csv"
ALL_TEAMS_CSV = DATA_DIR / "allteams_year_over_year.csv"
FAILED_REQUESTS_CSV = DATA_DIR / "failed_teamyearbyyear_requests.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
TEAM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

#print(f"Project root: {PROJECT_ROOT}")
#print(f"Data directory: {DATA_DIR}")

## Set up the NBA API team list

The first API step is to initialize the list of current NBA teams. `teams.get_teams()` returns 30 dictionaries, one for each team, and each dictionary includes a team ID. That team ID is what the `TeamYearByYearStats` endpoint needs in order to pull season-level statistics for a specific franchise.


In [ ]:
import nba_api

# get_teams returns a list of 30 dictionaries, one for each current NBA team.
# These static team dictionaries include the team IDs needed for the live stats endpoint.
nba_teams = teams.get_teams()
nba_teams[:3]



## Pull one team as a quick API check

Before pulling every team, I use the Charlotte Hornets as a small test case. This confirms that the team lookup, API request, retry logic, and CSV export are all working before the larger loop runs.

In [ ]:
# Find the Hornets dictionary using a case-insensitive match
hornets = [
    team for team in nba_teams
    if "hornets" in team["full_name"].lower()
][0]

print(hornets)


# Extract the team ID
hornets_id = hornets["id"]
print(f"Hornets ID: {hornets_id}")


# Fetch year-over-year team stats for the Hornets only
max_retries = 5
last_error = None

for attempt in range(1, max_retries + 1):
    try:
        print(f"Fetching Hornets stats: attempt {attempt} of {max_retries}")

        statsfinder = teamyearbyyearstats.TeamYearByYearStats(
            team_id=hornets_id,
            league_id="00",
            per_mode_simple="Totals",
            season_type_all_star="Regular Season",
            headers=STATS_HEADERS,
            timeout=120
        )

        hornets_stats = statsfinder.get_data_frames()[0]
        break

    except (ReadTimeout, ConnectionError) as e:
        last_error = e
        print(f"Attempt {attempt} failed: {e}")

        if attempt < max_retries:
            wait = 10 * attempt + random.uniform(2, 6)
            print(f"Waiting {wait:.1f} seconds before retrying...")
            time.sleep(wait)

else:
    raise RuntimeError("Failed to fetch Hornets stats after all retry attempts") from last_error


# Display and save to CSV in the project data folder
print(hornets_stats.head())

hornets_stats.to_csv(HORNETS_CSV, index=False)

print(f"Saved file: {HORNETS_CSV}")


### Inspect the Hornets dataset

These quick checks show the data types, column names, and team-name values returned by the endpoint. I used this to get acclimated with the data that would be returned and how things like team name changes would be handled before scaling the same endpoint across all teams.


In [ ]:
hornets_stats.dtypes

In [ ]:
hornets_stats.columns

In [ ]:
hornets_stats['TEAM_NAME'].unique()

# Getting our data

This step collects year-over-year team statistics for every NBA team using the `nba_api` package. The project setup above creates a local `data/` folder inside the same project folder as this notebook, so the file paths are not tied to my personal Downloads folder. If this project is moved, copied, or forked, the notebook should still write to the project-level `data/` folder as long as it is run from inside the project.

First, the list of NBA teams is pulled from `teams.get_teams()`. Then, for each team, the code checks whether that team’s year-over-year stats have already been saved in the local cache folder.

If a cached file already exists for a team, the notebook uses that saved file instead of sending another request to `stats.nba.com`. If the file does not exist yet, the notebook fetches the team’s data from the `TeamYearByYearStats` endpoint, adds identifying team metadata, and immediately saves the result as an individual team-level CSV file.

Because `stats.nba.com` can timeout or fail when many requests are sent back-to-back, this version includes several safeguards: NBA API headers, a longer request timeout, retry logic for failed requests, and a short randomized pause only after live requests. These changes make the data collection process more reliable and prevent one failed request from causing all previously collected data to be lost.

After all available team files are cached, the notebook reads the cached CSV files, concatenates them into one combined DataFrame called `allteams_year_over_year`, and saves the final dataset as `allteams_year_over_year.csv`. Any teams that still fail after the retry attempts are recorded separately in `failed_teamyearbyyear_requests.csv`, so the cell can be rerun later to retry only the missing teams.


In [ ]:

# Helper: team cache path
def get_team_cache_path(team):
    return TEAM_CACHE_DIR / f"{team['abbreviation']}_{team['id']}_teamyearbyyear.csv"


# Download one team if needed
def cache_team_year_by_year(team, max_retries=5):
    team_id = team["id"]
    abbreviation = team["abbreviation"]
    full_name = team["full_name"]

    cache_path = get_team_cache_path(team)

    # Skip API call if this team was already downloaded
    if cache_path.exists():
        print(f"Using cached file for {full_name}")
        return cache_path

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Fetching {full_name}: attempt {attempt} of {max_retries}")

            statsfinder = teamyearbyyearstats.TeamYearByYearStats(
                team_id=team_id,
                league_id="00",
                per_mode_simple="Totals",
                season_type_all_star="Regular Season",
                headers=STATS_HEADERS,
                timeout=120
            )

            df = statsfinder.get_data_frames()[0]

            # Add metadata from the static teams list
            df["STATIC_TEAM_ID"] = team_id
            df["STATIC_TEAM_ABBREVIATION"] = abbreviation
            df["STATIC_TEAM_FULL_NAME"] = full_name

            # Save immediately after a successful request
            df.to_csv(cache_path, index=False)

            print(f"Saved cached file for {full_name}")
            return cache_path

        except (ReadTimeout, ConnectionError) as e:
            last_error = e
            print(f"Failed for {full_name}: {e}")

            if attempt < max_retries:
                wait = 10 * attempt + random.uniform(2, 6)
                print(f"Waiting {wait:.1f} seconds before retrying...")
                time.sleep(wait)

    raise RuntimeError(f"Failed to fetch {full_name}") from last_error


# Cache all teams
failed_teams = []

for i, team in enumerate(nba_teams, start=1):
    print(f"\nTeam {i} of {len(nba_teams)}: {team['full_name']}")

    cache_exists_before = get_team_cache_path(team).exists()

    try:
        cache_team_year_by_year(team)

    except Exception as e:
        failed_teams.append({
            "team_id": team["id"],
            "abbreviation": team["abbreviation"],
            "full_name": team["full_name"],
            "error": str(e)
        })

    # Pause only if this team required a live NBA API attempt
    if i < len(nba_teams) and not cache_exists_before:
        pause = random.uniform(8, 14)
        print(f"Pausing {pause:.1f} seconds...")
        time.sleep(pause)


# Combine cached team files
cached_files = sorted(TEAM_CACHE_DIR.glob("*_teamyearbyyear.csv"))

if not cached_files:
    raise RuntimeError("No team files were downloaded successfully.")

allteams_year_over_year = pd.concat(
    [pd.read_csv(file) for file in cached_files],
    ignore_index=True
)

allteams_year_over_year.to_csv(ALL_TEAMS_CSV, index=False)

print(f"\nSaved combined file: {ALL_TEAMS_CSV}")
print(f"Rows: {len(allteams_year_over_year)}")
print(f"Columns: {len(allteams_year_over_year.columns)}")


# Report failures
if failed_teams:
    failed_df = pd.DataFrame(failed_teams)
    failed_df.to_csv(FAILED_REQUESTS_CSV, index=False)

    print("\nSome teams failed. Re-run this same cell to retry only missing teams.")
    display(failed_df)

else:
    print("\nAll teams downloaded successfully.")

### Preview the combined team dataset

This gives a quick look at the combined DataFrame after all cached team files have been read and stacked together.


In [ ]:
allteams_year_over_year.head()

### Check available columns

I list the columns here so it is clear which variables came from the API and which extra metadata columns were added during the caching step.

In [ ]:
list(allteams_year_over_year.columns)

# Data cleaning & analysis preparation

### Remove columns we don't need for this analysis

In [ ]:
allteams_year_over_year.drop(
    columns=[
        'GP',
        'CONF_RANK',
        'DIV_RANK',
        'PO_WINS',
        'PO_LOSSES',
        'CONF_COUNT',
        'DIV_COUNT',
        'NBA_FINALS_APPEARANCE',
        'FGM',
        'FGA',
        'FG_PCT',
        'FTM',
        'FTA',
        'FT_PCT',
        'OREB',
        'DREB',
        'REB',
        'AST',
        'PF',
        'STL',
        'TOV',
        'BLK',
        'PTS',
        'PTS_RANK',
        'STATIC_TEAM_ID',
        'STATIC_TEAM_ABBREVIATION',
        'STATIC_TEAM_FULL_NAME'
    ],
    inplace=True,
    errors='ignore'
)

### Reformat the YEAR column so that it takes the first 4 digits as this is the year the season began in and convert it to integers

In [ ]:
allteams_year_over_year['YEAR'] = allteams_year_over_year['YEAR'].str[:4].astype(int)


### Because the NBA did not add a three point line until the 1979 season, we want to get rid of all rows pretaining to years before 1979. We also want to remove any stats pertaining to the 2024 season because that is the season currently being played out

In [ ]:
# Remove rows with 'YEAR' older than 1979
allteams_year_over_year = allteams_year_over_year[allteams_year_over_year['YEAR'] > 1978]


# Remove any rows pretaining to 2024 season
allteams_year_over_year = allteams_year_over_year[allteams_year_over_year['YEAR'] < 2024]

### Create analysis variables

This cell creates total games played and per-game three-point makes and attempts. The per-game versions make the three-point volume variables easier to compare across seasons.

In [ ]:
allteams_year_over_year['total_games'] = allteams_year_over_year['WINS'] + allteams_year_over_year['LOSSES']

allteams_year_over_year['FG3M_pergame'] = allteams_year_over_year['FG3M'] / allteams_year_over_year['total_games']

allteams_year_over_year['FG3A_pergame'] = allteams_year_over_year['FG3A'] / allteams_year_over_year['total_games']

allteams_year_over_year.head()

### In older seasons closer to the introduction of the three point line, there are some instances of teams having 3 pointers made but none attempted. Because this makes up a small portion of our data ranging 1979-2024 these rows will be removed.

In [ ]:
# Remove rows with 0 three pointers attempted
allteams_year_over_year = allteams_year_over_year[allteams_year_over_year['FG3A'] > 0]


### Check the cleaned data after removing zero-attempt rows

This is a quick check to make sure the filter on three-point attempts worked as expected before moving into summary statistics and analysis.

In [ ]:
allteams_year_over_year.head()

# Getting preliminary summary statistics of data

### Getting the shape of the dataframe

In [ ]:
print(np.shape(allteams_year_over_year))

### Obtaining a summary of the dataframe

In [ ]:
allteams_year_over_year.info()

### Obtaining a statistical summary of the dataframe's numerical columns

In [ ]:
allteams_year_over_year.drop(columns=['TEAM_ID', 'YEAR']).describe()

### Addressing teams that have changed franchise names or locations in the dataframe

Some franchises changed names or moved cities during the time period covered by this dataset. For this project, I want each historical row to roll up to the current franchise identity. That makes the team labels easier to compare over time and avoids treating relocated or renamed versions of the same franchise as separate teams.


### Identify team-city combinations before standardizing franchise names

Before applying the franchise-name corrections, I print the unique city/name combinations so it is clear which historical labels need to be cleaned up.

In [ ]:
#Finding the combinations we must change
print(allteams_year_over_year[['TEAM_CITY', 'TEAM_NAME']].drop_duplicates())

### Standardize historical franchise labels

The mapping below converts older franchise names and locations to the modern franchise labels used in this analysis. This keeps the logic in one place instead of using many separate `loc` statements.


In [ ]:
# Clean whitespace first
allteams_year_over_year["TEAM_CITY"] = allteams_year_over_year["TEAM_CITY"].str.strip()
allteams_year_over_year["TEAM_NAME"] = allteams_year_over_year["TEAM_NAME"].str.strip()

# Map historical / relocated franchise names to the modern team labels used in this project
team_corrections = {
    ("New Orleans", "Hornets"): ("New Orleans", "Pelicans"),
    ("New Orleans/Oklahoma City", "Hornets"): ("New Orleans", "Pelicans"),
    ("New Orleans/Oklahoma City", "Pelicans"): ("New Orleans", "Pelicans"),

    ("Charlotte", "Bobcats"): ("Charlotte", "Hornets"),

    ("San Diego", "Clippers"): ("Los Angeles", "Clippers"),
    ("LA", "Clippers"): ("Los Angeles", "Clippers"),

    ("New Jersey", "Nets"): ("Brooklyn", "Nets"),

    ("Kansas City", "Kings"): ("Sacramento", "Kings"),

    ("Seattle", "SuperSonics"): ("Oklahoma City", "Thunder"),
    ("Oklahoma City", "SuperSonics"): ("Oklahoma City", "Thunder"),

    ("Vancouver", "Grizzlies"): ("Memphis", "Grizzlies"),

    ("Washington", "Bullets"): ("Washington", "Wizards"),
}

# Create current city/name pairs
current_pairs = pd.Series(
    list(zip(allteams_year_over_year["TEAM_CITY"], allteams_year_over_year["TEAM_NAME"])),
    index=allteams_year_over_year.index
)

# Look up corrected pairs where applicable
corrected_pairs = current_pairs.map(team_corrections)

# Update only rows that have a correction
mask = corrected_pairs.notna()

allteams_year_over_year.loc[mask, ["TEAM_CITY", "TEAM_NAME"]] = pd.DataFrame(
    corrected_pairs[mask].tolist(),
    index=allteams_year_over_year.index[mask],
    columns=["TEAM_CITY", "TEAM_NAME"]
)

### Confirm the franchise-name cleanup

After applying the mapping, I check the unique city/name combinations again to verify that the dataset now uses the intended 30 current team labels.

In [ ]:
#Checking that we are now left with the correct 30 team and city combinations
print(allteams_year_over_year[['TEAM_CITY', 'TEAM_NAME']].drop_duplicates())

# Now that data is cleaned can begin the EDA phase

### Correlation matrix for the main variables

The correlation matrix gives a first look at how the numerical variables relate to each other before fitting regression models. I remove duplicated or identifying variables so the matrix focuses on the analysis variables.

In [ ]:
correlation_columns_to_drop = [
    'WINS',
    'LOSSES',
    'FG3M',
    'FG3A',
    'TEAM_ID',
    'YEAR',
    'TEAM_CITY',
    'TEAM_NAME',
    'total_games'
]

correlation_matrix = (
    allteams_year_over_year
    .drop(columns=correlation_columns_to_drop, errors='ignore')
    .select_dtypes(include='number')
    .corr()
)

print(correlation_matrix)

### Scatterplot of three-point percentage and win percentage

This plot shows the relationship between team three-point shooting percentage and win percentage. The 1982 Los Angeles Lakers are highlighted because they stood out as the top-left outlier: a very strong winning team despite a very low three-point percentage.


In [ ]:
### Visualize correlation betweeen '3PT%' and 'WIN%' with scatterplot 

plt.figure(figsize=(12, 6))

# Make a string version of YEAR so this works whether YEAR is stored as 1982 or "1982-83"
year_as_string = allteams_year_over_year["YEAR"].astype(str)

lakers_outlier_mask = (
    year_as_string.str.startswith("1982") &
    (allteams_year_over_year["TEAM_CITY"] == "Los Angeles") &
    (allteams_year_over_year["TEAM_NAME"] == "Lakers")
)

warriors_outlier_mask = (
    year_as_string.str.startswith("2015") &
    (allteams_year_over_year["TEAM_CITY"] == "Golden State") &
    (allteams_year_over_year["TEAM_NAME"] == "Warriors")
)

# Original-style regplot
sns.regplot(
    data=allteams_year_over_year,
    x="FG3_PCT",
    y="WIN_PCT",
    color="blue",
    line_kws={"color": "orange"},
    scatter_kws={"alpha": 0.75}
)

# Overlay 1982 Los Angeles Lakers outlier in purple
lakers_outlier = allteams_year_over_year.loc[lakers_outlier_mask]

plt.scatter(
    lakers_outlier["FG3_PCT"],
    lakers_outlier["WIN_PCT"],
    color="purple",
    s=140,
    edgecolors="none",
    zorder=10,
    label="1982 Los Angeles Lakers"
)

# Overlay 2015 Golden State Warriors outlier in gold
warriors_outlier = allteams_year_over_year.loc[warriors_outlier_mask]

plt.scatter(
    warriors_outlier["FG3_PCT"],
    warriors_outlier["WIN_PCT"],
    color="gold",
    s=140,
    edgecolors="none",
    zorder=10,
    label="2015 Golden State Warriors"
)

plt.title("Distribution and Correlation between WIN% and 3P%")
plt.legend()

plt.savefig(
    FIGURES_DIR / "three_point_trends.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Data associated with the top left outlier: 1982 Los Angeles Lakers

In [ ]:
min_3PP_index = allteams_year_over_year['FG3_PCT'].idxmin()
print(allteams_year_over_year.loc[min_3PP_index])

Data associated with top right outlier: 2015 Golden State Warriors

In [ ]:
max_WP_index = allteams_year_over_year['WIN_PCT'].idxmax()
print(allteams_year_over_year.loc[max_WP_index])

### League-wide three-point trends over time

These plots show how three-point attempts, makes, and shooting percentage changed by season. This helps put the regression analysis in context because the value and usage of the three-point shot changed substantially over the period covered by the data.

In [ ]:
#Showing Changes in 3P%, 3PA, and 3PM over the course of the 1982 season through the 2023 season.
avg_3Pperc_perseason = allteams_year_over_year.groupby('YEAR')['FG3_PCT'].mean()
avg_3Patt_perseason = allteams_year_over_year.groupby('YEAR')['FG3A_pergame'].mean()
avg_3Pmade_perseason = allteams_year_over_year.groupby('YEAR')['FG3M_pergame'].mean()

#Initialize figure and 1x2 GridSpec
fig = plt.figure(figsize=(15,10))
gs = GridSpec(1,2, figure = fig)

# Set left plot for average 3P makes and attempts across all seasons
ax1 = fig.add_subplot(gs[0,0])
ax1.plot(np.unique(allteams_year_over_year['YEAR']),avg_3Patt_perseason, color='blue', label='Average 3-Pointers Attempted Per Game')
ax1.plot(np.unique(allteams_year_over_year['YEAR']),avg_3Pmade_perseason, color='red', label='Average 3-Pointers Made Per Game')
plt.xlabel("Season Year")
plt.ylabel("# of 3 Pointers Per game")
plt.title("Evolution of '3PA Per Game', '3PM Per Game' over the Course of 1982-2023 NBA Seasons", fontsize=10)
plt.legend()

#Set up right plot for average 3P percent across all seasons
ax2 = fig.add_subplot(gs[0,1])
ax2.plot(np.unique(allteams_year_over_year['YEAR']),avg_3Pperc_perseason, color='black', label='Average 3-Point Shooting %')
plt.xlabel("Season Year")
plt.ylabel("Shooting Percent")
plt.title("Evolution of 3-Point shooting Percentage over the Course of 1982-2023 NBA Seasons", fontsize=10)
plt.legend()

plt.savefig(
    FIGURES_DIR / "win_pct_vs_fg3_pct.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Adding some context for the above graphs:

We see that, upon implementing the 3 point shot, initially there is a gradual increase in average 3 point attempts/makes per game across the league; this is because there was not an immediate adjustment to an optimized play style after the addition of the 3 point line because both players and coaches were not eager to shift away from the inside scoring style of play that had defined winning basketball for so long when there was no incentive to shoot from further away.

After the gradual increase, shortly after 1990, we see a spike in both of these followed by a sharp drop off. This is because in 1994 the NBA identified that there had been a steady decline in pace of play and scoring across the league. Because the NBA is a business they viewed this as their product losing value as it is less "exciting". In hopes of neautralizing this trend they decided to incentize more three point shooting by shortening the 3 point line to 22 feet around the entire arc from 1994 to 1996 when normally it was 23 feet and nine inches across the arc and 22 feet only in the corners. This resulted in some record breaking 3 point shooting performances by indiviudal players during that span along with overall better 3 point shooting across the league, but the scoring trend did not improve as the NBA hoped. This is why they reverted to the original distance after the three year span which explains the fall off after the spike.

The flatenning of the curve around 2010, followed by the continuous rise in 3 point shooting year over coincides with Stephen Curry's breakout season in 2012 when he broke the record for three pointers made in a season with 272 which he went on to break again many times. Stephen Curry and his Warriors began to put the league on notice that optimizing their offensive gameplan around a high pace and focusing on shooting threes could be truly over powering. Because of this in my regression analysis I will initially make my model using the entire dataset from 1982-2023 and then repeat the analysis only including years 2010-2023 in the dataset to make the model. Filtering the dataset to only include a more recent time frame should result in the regression model improving as it will be based only on data that most accurately reflects the current state of the game.

# Now that we have explored the data, can continue to the analysis phase

### Historical regression: win percentage on three-point percentage

This first regression uses the full cleaned dataset. The goal is to estimate the overall historical relationship between three-point field goal percentage and season win percentage.


In [ ]:
#Win % on 3P attempts
y = allteams_year_over_year['WIN_PCT'].to_numpy()
x = allteams_year_over_year['FG3_PCT'].to_numpy()

# Split data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

#Initialize and fit linear regression
WinPerc_on3Patt = LinearRegression().fit(x_train.reshape(-1,1), y_train)

# Coefficients (slope(s) for each feature)
print("Coefficients:", WinPerc_on3Patt.coef_)

# Intercept (y-intercept of the regression line)
print("Intercept:", WinPerc_on3Patt.intercept_)

# Make predictions
y_pred = WinPerc_on3Patt.predict(x_test.reshape(-1,1))

# Models R square
r2 = WinPerc_on3Patt.score(x_test.reshape(-1,1),y_test)
print("R-squared value:", r2)

mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Absolute Error: {mae}')

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f'Root Mean Squared Error: {rmse}')

### Modern-era regression: 2010 through 2023

The second regression repeats the same model after filtering to the modern NBA era. This is useful because three-point shooting became much more central to offensive strategy after 2010, so the relationship may be stronger in this period.

In [ ]:
# Remove rows with 'YEAR' older than 1979
allteams_year_over_year = allteams_year_over_year[allteams_year_over_year['YEAR'] > 2009]

# Remove any rows pretaining to 2024 season
allteams_year_over_year = allteams_year_over_year[allteams_year_over_year['YEAR'] < 2024]

#Win % on 3P attempts
y = allteams_year_over_year['WIN_PCT'].to_numpy()
x = allteams_year_over_year['FG3_PCT'].to_numpy()

# Split data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

#Initialize and fit linear regression
WinPerc_on3Patt = LinearRegression().fit(x_train.reshape(-1,1), y_train)

# Coefficients (slope(s) for each feature)
print("Coefficients:", WinPerc_on3Patt.coef_)

# Intercept (y-intercept of the regression line)
print("Intercept:", WinPerc_on3Patt.intercept_)

# Make predictions
y_pred = WinPerc_on3Patt.predict(x_test.reshape(-1,1))

# Models R square
r2 = WinPerc_on3Patt.score(x_test.reshape(-1,1),y_test)
print("R-squared value:", r2)

mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Absolute Error: {mae}')

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f'Root Mean Squared Error: {rmse}')

# Conclusion:

Coefficients: In the historical regression, we have a coefficient of .877 suggesting that a 1% increase in 3 point shooting percentage is associated with a .877% increase in season win percentage. When filtering for the modern NBA era, we have a coefficient of 4.742 which suggests that a 1% increase in 3 point shooting percentage is associated with a 4.742% increase in season win percentage. This supports a stronger and more direct relationship between 3 point shooting and winning in the modern NBA era.

Intercepts: The historical model had an intercept of .201 suggesting that hypothetically, if a team had a 3 point percentage of 0 we would predict that they have a 20.1 win percentage on that season. The modern era's model intercept of -1.196 is a bit unrealistic in interpretation because no team can ever have a negative win percentage, but this does reflect how heavily win rates in the modern NBA era rely on 3 point shooting.

Model Fit: The historical model had an R sqaure value of .077 indicating that 7.7% of the variation in win percentage is explained by 3 point shooting percentage historically. The modern era's model R square made a large improvement to .321 which again supports that the modern NBA era places more focus on 3 point shooting in winning strategies.

Prediction Accuracy: Given that win percentage ranges from 0-1, the historical model's RMSE of .144 means that the model's predictions typically deviate by about 14.4% from the actual values. The modern era's model RMSE decreases to .120 indicating improved accuracy in predicting win percentage based on three point shooting percentage in more recent years. This supports, as the 3 point revolution has occured, the correlation between three point shooting and winning has increased.